In [ ]:
# LS07 - Code Assistant Chatbot
# Dataset: CodeSearchNet
# Goal: explain and assist with code-related queries.

import sys
import subprocess

subprocess.run([sys.executable, "-m", "pip", "install", "transformers", "datasets", "evaluate", "pandas", "scikit-learn", "-q"], check=True)

In [ ]:
import copy
import re
import torch
import pandas as pd
from datasets import load_dataset, Dataset
from sklearn.metrics import confusion_matrix, classification_report
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
)

MODEL_NAME = "distilgpt2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device: {DEVICE}")

In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id
model.to(DEVICE)

base_model = copy.deepcopy(model).to(DEVICE)
base_model.eval()

In [ ]:
raw_train = load_dataset("code-search-net/code_search_net", "python", split="train[:600]")
raw_eval = load_dataset("code-search-net/code_search_net", "python", split="test[:40]")

print(raw_train)
print(raw_train.column_names)

In [ ]:
def clean_text(text, max_chars=700):
    if text is None:
        return ""
    text = re.sub(r"\s+", " ", str(text)).strip()
    return text[:max_chars]


def clean_code(code, max_chars=900):
    if code is None:
        return ""
    code = str(code).replace("\t", "    ").strip()
    return code[:max_chars]


def make_training_example(row):
    doc = clean_text(row.get("func_documentation_string", ""), max_chars=500)
    code = clean_code(row.get("func_code_string", ""), max_chars=700)

    if not doc or not code:
        return None

    return (
        "You are a helpful code assistant. Answer clearly and focus on code behavior.\n"
        f"User: Explain what this Python function does.\n```python\n{code}\n```\n"
        f"Bot: {doc}"
    )


def make_query(row):
    code = clean_code(row.get("func_code_string", ""), max_chars=700)
    return (
        "You are a helpful code assistant. Answer clearly and focus on code behavior.\n"
        f"User: Explain what this Python function does.\n```python\n{code}\n```\n"
        "Bot:"
    )


def make_expected(row):
    return clean_text(row.get("func_documentation_string", ""), max_chars=500)

train_texts = [make_training_example(row) for row in raw_train]
train_texts = [text for text in train_texts if text]

train_dataset = Dataset.from_dict({"text": train_texts})
print(train_dataset)
print(train_dataset[0]["text"][:1000])

In [ ]:
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=384,
    )

tokenized_train = train_dataset.map(tokenize, remove_columns=["text"])

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

In [ ]:
training_args = TrainingArguments(
    output_dir="./results_code_assistant_chatbot",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    max_steps=120,
    learning_rate=5e-5,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    dataloader_pin_memory=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    data_collator=data_collator,
)

trainer.train()

In [ ]:
def generate_response(prompt, model_to_use, max_new_tokens=90):
    model_to_use.eval()
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=384,
    ).to(DEVICE)

    with torch.no_grad():
        output = model_to_use.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=max_new_tokens,
            min_new_tokens=8,
            do_sample=True,
            top_p=0.9,
            top_k=50,
            temperature=0.4,
            repetition_penalty=1.2,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output[0][inputs["input_ids"].shape[-1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    response = response.split("User:")[0].split("Bot:")[0].strip()
    return response

In [ ]:
eval_rows = []
for row in raw_eval:
    expected = make_expected(row)
    query = make_query(row)
    if expected and "```python" in query:
        eval_rows.append({"query": query, "expected": expected})
    if len(eval_rows) == 12:
        break

for i, item in enumerate(eval_rows[:3], 1):
    before = generate_response(item["query"], base_model)
    after = generate_response(item["query"], model)
    print(f"Example {i}")
    print("QUERY:", item["query"].split("User:", 1)[-1][:350].strip())
    print("BEFORE:", before)
    print("AFTER:", after)
    print("-" * 80)

In [ ]:
CODE_TERMS = {
    "function", "return", "argument", "parameter", "value", "list", "dict", "string",
    "object", "class", "method", "file", "path", "data", "array", "loop", "exception",
    "python", "input", "output", "call", "variable", "module", "decorator", "iterator",
}


def normalize_words(text):
    return set(re.findall(r"[a-zA-Z_]{3,}", text.lower()))


def keyword_overlap(expected, predicted):
    expected_words = normalize_words(expected)
    predicted_words = normalize_words(predicted)
    if not expected_words or not predicted_words:
        return 0.0
    return len(expected_words & predicted_words) / max(1, len(expected_words))


def relevance_score(expected, predicted):
    overlap = keyword_overlap(expected, predicted)
    return min(1.0, overlap * 3.0)


def coherence_score(predicted):
    words = predicted.split()
    if len(words) < 6:
        return 0.2 if words else 0.0
    unique_ratio = len(set(words)) / len(words)
    if unique_ratio < 0.35:
        return 0.3
    if predicted.count(".") + predicted.count(",") + predicted.count(";") >= 1:
        return 0.9
    return 0.7


def fluency_score(predicted):
    words = predicted.split()
    if len(words) < 5:
        return 0.2 if words else 0.0
    alpha_words = sum(any(ch.isalpha() for ch in word) for word in words)
    return min(1.0, alpha_words / max(1, len(words)))


def domain_accuracy_score(expected, predicted):
    predicted_words = normalize_words(predicted)
    expected_words = normalize_words(expected)
    code_hits = len(predicted_words & CODE_TERMS)
    expected_hits = len(predicted_words & expected_words)
    return min(1.0, (0.18 * code_hits) + (0.12 * expected_hits))


def evaluate_response(expected, predicted):
    return {
        "Relevance": relevance_score(expected, predicted),
        "Coherence": coherence_score(predicted),
        "Fluency": fluency_score(predicted),
        "Domain Accuracy": domain_accuracy_score(expected, predicted),
    }


def quality_label(scores):
    avg = sum(scores.values()) / len(scores)
    if avg >= 0.70:
        return "GOOD"
    if avg >= 0.40:
        return "PARTIAL"
    if avg > 0.05:
        return "WRONG"
    return "EMPTY"

In [ ]:
records = []

for item in eval_rows:
    before = generate_response(item["query"], base_model)
    after = generate_response(item["query"], model)
    before_scores = evaluate_response(item["expected"], before)
    after_scores = evaluate_response(item["expected"], after)

    records.append({
        "expected": item["expected"],
        "before": before,
        "after": after,
        "before_label": quality_label(before_scores),
        "after_label": quality_label(after_scores),
        **{f"before_{k}": v for k, v in before_scores.items()},
        **{f"after_{k}": v for k, v in after_scores.items()},
    })

results = pd.DataFrame(records)

performance_rows = []
for metric in ["Relevance", "Coherence", "Fluency", "Domain Accuracy"]:
    performance_rows.append({
        "Metric": metric,
        "Before": round(results[f"before_{metric}"].mean(), 3),
        "After": round(results[f"after_{metric}"].mean(), 3),
    })

performance_table = pd.DataFrame(performance_rows)
performance_table

In [ ]:
labels = ["EMPTY", "WRONG", "PARTIAL", "GOOD"]
confusion = pd.DataFrame(
    confusion_matrix(results["before_label"], results["after_label"], labels=labels),
    index=[f"Before_{label}" for label in labels],
    columns=[f"After_{label}" for label in labels],
)

print("Confusion analysis: movement from BEFORE quality to AFTER quality")
confusion

In [ ]:
print(classification_report(
    results["before_label"],
    results["after_label"],
    labels=labels,
    zero_division=0,
))

In [ ]:
display_columns = ["before_label", "after_label", "expected", "before", "after"]
results[display_columns].head(8)

In [ ]:
print("Code Assistant Chatbot. Type 'quit' to exit.\n")

while True:
    user_input = input("User: ")

    if user_input.lower().strip() == "quit":
        print("Bot: Goodbye.")
        break

    prompt = (
        "You are a helpful code assistant. Answer clearly and focus on code behavior.\n"
        f"User: {user_input}\n"
        "Bot:"
    )
    response = generate_response(prompt, model)
    print("Bot:", response)